In [1]:
from src.basemodel import Capa, Clasificacion,Familia,Articulo,Segmento,Clase,Recomendacion
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
import psycopg2
from src.database import create_connection
from src.ai import generate_prompt,generar_familias_prompt, generar_clases_prompt, generar_productos_prompt
import json

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_IA_MODEL = os.getenv("GEMINI_IA_MODEL")

client = genai.Client(api_key=GEMINI_API_KEY)

segmentos:list[Segmento] = []
articulos: list[Articulo] = []
familias: list[Familia] = []
clases:list[Clase] = []

capa1_segmentos:dict[str,Capa[Segmento]] = {}
capa2_familias:dict[str,Capa[Familia]] = {}
capa3_clases:dict[str,Capa[Clase]] = {}
capa4_productos:dict[str,Capa[Articulo]] = {}

catalogo: dict[str,list[Familia]] = []

articulos = [
    Articulo("adajk","coca cola 1/2"),
    Articulo("calmkllwa","gaseosa kr 300 ml"),
    Articulo("aadac","cuarto de pollo"),
    Articulo("alwekmav","delivery domicilio"),
    Articulo("acmwakoi","chaufa + pollo"),
    Articulo("aaasdc","tallarin saltado")
]


In [2]:
#traer segmentos
conn = create_connection()
with conn.cursor() as cur:
    cur.execute("select id,descripcion from segmento")
    result = cur.fetchall()
conn.close()
segmentos = [Segmento(id,nombre) for id,nombre in result]

In [3]:
##generar segmentos
prompt = generate_prompt(
        (seg.nombre for seg in segmentos),
        (art.nombre for art in articulos)
    )
#print(prompt)
response = client.models.generate_content(
    contents=prompt,
    model=GEMINI_IA_MODEL,
    config={
        "response_mime_type":"application/json",
        "response_schema":list[Recomendacion]
    }
)
data: list[Recomendacion] = response.parsed

In [4]:
for clasificacion in data:
    seg = segmentos[clasificacion.id_grupo]
    art = articulos[clasificacion.id_articulo]
    print("-------------")
    print(f"articulo: {art.nombre}, segmento: {seg.nombre} ({seg.id})")
    print(f"descripcion: {clasificacion.descripcion_detallada_articulo}")
    print(f"motivo: {clasificacion.motivo} ({clasificacion.confianza})")

-------------
articulo: coca cola 1/2, segmento: Alimentos, Bebidas y Tabaco  (50000000)
descripcion: Bebida gaseosa de sabor cola en presentación personal de 500 mililitros.
motivo: Se clasifica en Alimentos, Bebidas y Tabaco por ser un producto de consumo directo. (1.0)
-------------
articulo: gaseosa kr 300 ml, segmento: Alimentos, Bebidas y Tabaco  (50000000)
descripcion: Bebida gaseosa de diversas variedades en presentación pequeña de 300 mililitros.
motivo: Se clasifica en Alimentos, Bebidas y Tabaco por ser un producto de consumo directo. (1.0)
-------------
articulo: cuarto de pollo, segmento: Alimentos, Bebidas y Tabaco  (50000000)
descripcion: Plato de comida consistente en una porción de pollo asado equivalente a un cuarto de unidad, generalmente acompañado de papas fritas y ensalada.
motivo: Se clasifica en Alimentos, Bebidas y Tabaco por ser una preparación alimenticia. (0.95)
-------------
articulo: delivery domicilio, segmento: Servicios de Viajes, Alimentación, Alojamie

In [5]:
##crear capa1 segmentos
for match in data:
    indexSegmento:int = match.id_grupo
    segmento:Familia = segmentos[indexSegmento]
    articulo:Articulo = articulos[match.id_articulo]
    capa1_segmentos.setdefault(segmento.id,Capa[Familia](segmento,indexSegmento))
    capa1_segmentos.get(segmento.id).items.append(articulo)

In [6]:
#obtener familias
segmento_ids = tuple(id[:2] for id in capa1_segmentos.keys())
query = """ select id,descripcion from familia where {} """.format(" OR ".join(["id like %s"] * len(segmento_ids)))
params = tuple(f"{id}%" for id in segmento_ids)
conn = create_connection()
with conn.cursor() as cur:
    cur.execute(query,params)
    result = cur.fetchall()
conn.close()
familias = [Familia(id,nombre) for id,nombre in result]

In [7]:
#agregar familias a segmentos
for capa in capa1_segmentos.values(): capa.value.familias = []
for fam in familias:
    tramoSegmento = fam.id[:2]
    segmento = next((s.value for s in capa1_segmentos.values() if s.value.id.startswith(tramoSegmento)),None)
    if segmento is not None: segmento.familias.append(fam)

In [8]:
#generar familias
datas: dict[str,list[Clasificacion]] = {}
for seg in capa1_segmentos.values():
    prompt = generar_familias_prompt(
        [fam.nombre for fam in seg.value.familias],
        [art.nombre for art in seg.items],
    )
    #print(prompt)
    response = client.models.generate_content(
        contents=prompt,
        model=GEMINI_IA_MODEL,
        config={
            "response_mime_type":"application/json",
            "response_schema":list[Clasificacion]
        }
    )
    datas[seg.value.id] = response.parsed

In [9]:
capas_actuales = capa1_segmentos
for idCapa, listaClasificacion in datas.items():
    for clasificacion in listaClasificacion:
        capa = capas_actuales.get(idCapa)
        grupo = capa.value.familias[clasificacion.id_grupo]
        art = capa.items[clasificacion.id_articulo]
        print("-------------")
        print(f"articulo: {art.nombre}, familia: {grupo.nombre} ({grupo.id})")
        print(f"motivo: {clasificacion.motivo} ({clasificacion.confianza})")

-------------
articulo: coca cola 1/2, familia: Bebidas (50200000)
motivo: La coca cola es una bebida gaseosa comercial. (1.0)
-------------
articulo: gaseosa kr 300 ml, familia: Bebidas (50200000)
motivo: La gaseosa es un producto clasificado como bebida. (1.0)
-------------
articulo: cuarto de pollo, familia: Alimentos preparados y conservados (50190000)
motivo: El cuarto de pollo es un plato preparado listo para el consumo. (0.95)
-------------
articulo: chaufa + pollo, familia: Alimentos preparados y conservados (50190000)
motivo: El arroz chaufa es un alimento preparado compuesto. (0.95)
-------------
articulo: tallarin saltado, familia: Alimentos preparados y conservados (50190000)
motivo: El tallarín saltado es una comida preparada en restaurante. (0.95)
-------------
articulo: delivery domicilio, familia: Restaurantes y catering (servicios de comidas y bebidas) (90100000)
motivo: El servicio de delivery de alimentos corresponde a la distribución y entrega de comidas preparadas,

In [10]:
#crear capa2_familias
for idSegmento,res in datas.items():
    capaSegmento: Capa[Segmento] = capa1_segmentos.get(idSegmento,None)
    if capaSegmento is None: continue
    for clasificacion in res:
        familia = capaSegmento.value.familias[clasificacion.id_grupo]
        articulo = capaSegmento.items[clasificacion.id_articulo]
        if familia is None or articulo is None: continue
        if capaSegmento.subcapas.get(familia.id) is None:
            capa = Capa[Familia](familia,clasificacion.id_grupo)
            capaSegmento.subcapas[familia.id] = capa
            capa2_familias[familia.id] = capa
        capaFamilia: Capa[Familia] = capaSegmento.subcapas.get(familia.id)
        capaFamilia.items.append(articulo)

In [11]:
#obtener clases
familia_ids = tuple(id[:4] for id in capa2_familias.keys())
query = """ select id,descripcion from clase where {} """.format(" OR ".join(["id like %s"] * len(familia_ids)))
params = tuple(f"{id}%" for id in familia_ids)
conn = create_connection()
with conn.cursor() as cur:
    cur.execute(query,params)
    result = cur.fetchall()
conn.close()
clases = [Clase(id,nombre) for id,nombre in result]

In [12]:
#agregar clases a familias
for capa in capa2_familias.values(): capa.value.clases = []
for cla in clases:
    tramoFamilia = cla.id[:4]
    familia = next((s.value for s in capa2_familias.values() if s.value.id.startswith(tramoFamilia)),None)
    if familia is not None: familia.clases.append(cla)

In [13]:
#generar clases
datas: dict[str,list[Clasificacion]] = {}
for seg in capa2_familias.values():
    prompt = generar_clases_prompt(
        [cla.nombre for cla in seg.value.clases],
        [art.nombre for art in seg.items],
    )
    #print(prompt)
    response = client.models.generate_content(
        contents=prompt,
        model=GEMINI_IA_MODEL,
        config={
            "response_mime_type":"application/json",
            "response_schema":list[Clasificacion]
        }
    )
    datas[seg.value.id] = response.parsed

In [14]:
capas_actuales = capa2_familias
for idCapa, listaClasificacion in datas.items():
    for clasificacion in listaClasificacion:
        capa = capas_actuales.get(idCapa)
        grupo = capa.value.clases[clasificacion.id_grupo]
        art = capa.items[clasificacion.id_articulo]
        print("-------------")
        print(f"articulo: {art.nombre}, clase: {grupo.nombre} ({grupo.id})")
        print(f"motivo: {clasificacion.motivo} ({clasificacion.confianza})")

-------------
articulo: coca cola 1/2, clase: Bebidas no alcohólicas (50202300)
motivo: Coca cola es una bebida gaseosa no alcohólica (1.0)
-------------
articulo: gaseosa kr 300 ml, clase: Bebidas no alcohólicas (50202300)
motivo: Gaseosa KR es una bebida gaseosa no alcohólica (1.0)
-------------
articulo: cuarto de pollo, clase: Platos combinados empaquetados (50192700)
motivo: El cuarto de pollo es un plato principal preparado que suele acompañarse con otros elementos, encajando en platos combinados. (0.95)
-------------
articulo: chaufa + pollo, clase: Platos combinados empaquetados (50192700)
motivo: El arroz chaufa con pollo es una preparación compuesta que se clasifica como plato combinado. (0.98)
-------------
articulo: tallarin saltado, clase: Pasta o tallarines natural (50192900)
motivo: El tallarín saltado es un plato cuya base principal es la pasta o tallarines. (0.96)
-------------
articulo: delivery domicilio, clase: Servicios de comida para llevar y a domicilio (90101800

In [15]:
#crear capa3 clases
for idFamilia,res in datas.items():
    capaFamilia: Capa[Familia] = capa2_familias.get(idFamilia,None)
    if capaFamilia is None: continue
    for clasificacion in res:
        clase = capaFamilia.value.clases[clasificacion.id_grupo]
        articulo = capaFamilia.items[clasificacion.id_articulo]
        if clase is None and articulo is None: continue
        if capaFamilia.subcapas.get(clase.id) is None:
            capa = Capa[Clase](clase,clasificacion.id_grupo)
            capaFamilia.subcapas[clase.id] = capa
            capa3_clases[clase.id] = capa
        capaClase: Capa[Clase] = capaFamilia.subcapas.get(clase.id)
        capaClase.items.append(articulo)

In [16]:
for capa in capa3_clases.values():
    for articulo in capa.items:
        print( f"{capa.value.id}:{capa.value.nombre}, {articulo.id}:{articulo.nombre}")

50202300:Bebidas no alcohólicas, adajk:coca cola 1/2
50202300:Bebidas no alcohólicas, calmkllwa:gaseosa kr 300 ml
50192700:Platos combinados empaquetados, aadac:cuarto de pollo
50192700:Platos combinados empaquetados, acmwakoi:chaufa + pollo
50192900:Pasta o tallarines natural, aaasdc:tallarin saltado
90101800:Servicios de comida para llevar y a domicilio, alwekmav:delivery domicilio


In [17]:
#obtener productos de BD
articulos_id = tuple(id[:4] for id in capa3_clases.keys())
query = """ select id,descripcion from producto where {} """.format(" OR ".join(["id like %s"] * len(articulos_id)))
params = tuple(f"{id}%" for id in articulos_id)
conn = create_connection()
with conn.cursor() as cur:
    cur.execute(query,params)
    result = cur.fetchall()
conn.close()
productos = [Articulo(id,nombre) for id,nombre in result]

In [18]:
#agregar productos a clases
for capa in capa3_clases.values(): capa.value.productos = []
for pro in productos:
    tramoClase = pro.id[:6]
    clase = next((s.value for s in capa3_clases.values() if s.value.id.startswith(tramoClase)),None)
    if clase is not None: clase.productos.append(pro)

In [19]:
#generar productos
datas: dict[str,list[Clasificacion]] = {}
for cla in capa3_clases.values():
    prompt = generar_productos_prompt(
        [pro.nombre for pro in cla.value.productos],
        [art.nombre for art in cla.items],
    )
    #print(prompt)
    response = client.models.generate_content(
        contents=prompt,
        model=GEMINI_IA_MODEL,
        config={
            "response_mime_type":"application/json",
            "response_schema":list[Clasificacion]
        }
    )
    datas[cla.value.id] = response.parsed

In [20]:
capas_actuales = capa3_clases
for idCapa, listaClasificacion in datas.items():
    for clasificacion in listaClasificacion:
        capa = capas_actuales.get(idCapa)
        grupo = capa.value.productos[clasificacion.id_grupo]
        art = capa.items[clasificacion.id_articulo]
        print("-------------")
        print(f"articulo: {art.nombre}, producto: {grupo.nombre} ({grupo.id})")
        print(f"motivo: {clasificacion.motivo} ({clasificacion.confianza})")

-------------
articulo: coca cola 1/2, producto: Refrescos (50202306)
motivo: Las gaseosas son bebidas carbonatadas que se clasifican como refrescos dentro de las categorías de consumo habitual en restaurantes. (0.95)
-------------
articulo: gaseosa kr 300 ml, producto: Refrescos (50202306)
motivo: Las gaseosas comerciales de bajo volumen se clasifican bajo la categoría de refrescos. (0.95)
-------------
articulo: cuarto de pollo, producto: Comidas combinadas frescas (50192701)
motivo: El cuarto de pollo es una comida preparada y servida fresca en el restaurante (0.98)
-------------
articulo: chaufa + pollo, producto: Comidas combinadas frescas (50192701)
motivo: El arroz chaufa con pollo es una comida combinada preparada al momento y servida fresca (0.98)
-------------
articulo: tallarin saltado, producto: Pasta sencilla o fideos (50192901)
motivo: El tallarin saltado es un plato preparado que utiliza pasta sencilla para su consumo inmediato, propio de un restaurante. (0.95)
---------

In [21]:
#crear capa4 productos
for idClase,res in datas.items():
    capaClase: Capa[Clase] = capa3_clases.get(idClase,None)
    if capaClase is None: continue
    for clasificacion in res:
        producto = capaClase.value.productos[clasificacion.id_grupo]
        articulo = capaClase.items[clasificacion.id_articulo]
        if producto is None and articulo is None: continue
        if capaClase.subcapas.get(producto.id) is None:
            capa = Capa[Articulo](producto,clasificacion.id_grupo)
            capaClase.subcapas[producto.id] = capa
            capa4_productos[producto.id] = capa
        capaArticulo: Capa[Clase] = capaClase.subcapas.get(producto.id)
        capaArticulo.items.append(articulo)

In [22]:
for idProducto,capa in capa4_productos.items():
    capaProducto = capa4_productos.get(idProducto)
    print(capaProducto.value.id,capaProducto.value.nombre,"--------------")
    print([item.nombre for item in capaProducto.items][:])

50202306 Refrescos --------------
['coca cola 1/2', 'gaseosa kr 300 ml']
50192701 Comidas combinadas frescas --------------
['cuarto de pollo', 'chaufa + pollo']
50192901 Pasta sencilla o fideos --------------
['tallarin saltado']
90101802 Servicios de comidas a domicilio --------------
['delivery domicilio']
